# Milestone 1: Data Profiling — Titik Panas (Hotspot) Kebakaran Hutan dan Lahan Indonesia

**Dataset:** Indonesia Fire Hotspots (NASA FIRMS, VIIRS 375 m), 2012–2026 — detail sumber dan lisensi ada di `data/README.md`.

**Pertanyaan analisis yang akan dijawab sampai Milestone 3:**
1. Kapan musim kebakaran terjadi dan tahun mana yang paling parah?
2. Provinsi/pulau mana yang paling sering terbakar, dan apakah polanya berubah dari tahun ke tahun?
3. Bagaimana intensitas api (FRP) berbeda antar wilayah, waktu siang/malam, dan musim?

Notebook ini memprofilkan data mentah **tanpa memuat seluruh file ke RAM**: Polars `scan_parquet` (lazy) dan DuckDB yang membaca Parquet langsung dari disk.

In [1]:
import sys
import time
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
sys.path.insert(0, str(ROOT))

import duckdb
import polars as pl

from src.config import RAW_PATH

pl.Config.set_tbl_rows(40)
pl.Config.set_tbl_cols(14)

if not RAW_PATH.exists():
    raise FileNotFoundError(
        f"Dataset tidak ditemukan: {RAW_PATH}. Ikuti petunjuk di data/README.md."
    )

print(f"Polars {pl.__version__} | DuckDB {duckdb.__version__}")
print(f"Dataset    : {RAW_PATH.relative_to(ROOT)}")
print(f"Ukuran file: {RAW_PATH.stat().st_size / 1_000_000:.1f} MB (Parquet terkompresi)")

Polars 2.0.0 | DuckDB 1.5.6
Dataset    : data/raw/hotspots.parquet
Ukuran file: 133.4 MB (Parquet terkompresi)


## 1. Skema dan Jumlah Baris (Polars Lazy)

`scan_parquet` hanya membaca metadata. Skema dan jumlah baris bisa diketahui tanpa membaca isi kolom.

In [2]:
lazy_data = pl.scan_parquet(RAW_PATH)
schema = lazy_data.collect_schema()
schema

Schema([('acq_date', Datetime(time_unit='ns', time_zone=None)),
        ('acq_datetime', Datetime(time_unit='ns', time_zone=None)),
        ('satellite', String),
        ('latitude', Float64),
        ('longitude', Float64),
        ('province', String),
        ('confidence', String),
        ('frp', Float64),
        ('bright_ti4', Float64),
        ('bright_ti5', Float64),
        ('daynight', String),
        ('processing', String)])

In [3]:
jumlah_baris = lazy_data.select(pl.len()).collect().item()
print(f"Jumlah baris : {jumlah_baris:,}")
print(f"Jumlah kolom : {len(schema)}")

Jumlah baris : 6,087,130
Jumlah kolom : 12


**Kamus data** (berdasarkan dokumentasi NASA FIRMS VIIRS dan halaman dataset):

| Kolom | Arti |
|---|---|
| `acq_date`, `acq_datetime` | Tanggal dan waktu lintasan satelit (**UTC**) |
| `satellite` | `Suomi NPP` (sejak 2012) atau `NOAA-20` (sejak 2018) |
| `latitude`, `longitude` | Pusat piksel 375 m yang terdeteksi panas |
| `province` | Provinsi (nama bahasa Inggris, batas geoBoundaries 34 provinsi) |
| `confidence` | Tingkat keyakinan deteksi: `low` / `nominal` / `high` |
| `frp` | *Fire Radiative Power* (MW): intensitas api |
| `bright_ti4`, `bright_ti5` | Suhu kecerahan kanal I4 (3,74 µm) dan I5 (11,45 µm), Kelvin |
| `daynight` | Lintasan siang (`D`, ±13.30 waktu lokal) atau malam (`N`, ±01.30) |
| `processing` | `SP` = arsip final, `NRT` = near-real-time (sementara) |

In [4]:
lazy_data.head(5).collect()

acq_date,acq_datetime,satellite,latitude,longitude,province,confidence,frp,bright_ti4,bright_ti5,daynight,processing
datetime[ns],datetime[ns],str,f64,f64,str,str,f64,f64,f64,str,str
2018-04-01 00:00:00,2018-04-01 04:16:00,"""NOAA-20""",-4.01907,136.07845,"""Papua""","""low""",34.34,337.23,298.31,"""D""","""SP"""
2018-04-01 00:00:00,2018-04-01 04:16:00,"""NOAA-20""",-4.01018,136.09189,"""Papua""","""low""",9.14,333.0,295.87,"""D""","""SP"""
2018-04-01 00:00:00,2018-04-01 04:16:00,"""NOAA-20""",-3.98694,136.09912,"""Papua""","""nominal""",6.11,332.31,294.42,"""D""","""SP"""
2018-04-01 00:00:00,2018-04-01 04:16:00,"""NOAA-20""",-3.85214,136.35695,"""Papua""","""low""",3.93,330.34,297.54,"""D""","""SP"""
2018-04-01 00:00:00,2018-04-01 05:56:00,"""NOAA-20""",-7.7919,112.01424,"""East Java""","""nominal""",6.58,333.0,299.4,"""D""","""SP"""


## 2. Kelengkapan: Jumlah Null per Kolom

In [5]:
null_per_kolom = (
    lazy_data.select(pl.all().null_count())
    .collect()
    .transpose(include_header=True, header_name="kolom", column_names=["jumlah_null"])
    .with_columns((pl.col("jumlah_null") / jumlah_baris * 100).round(3).alias("persen_null"))
)
null_per_kolom

kolom,jumlah_null,persen_null
str,u32,f64
"""acq_date""",0,0.0
"""acq_datetime""",0,0.0
"""satellite""",0,0.0
"""latitude""",0,0.0
"""longitude""",0,0.0
"""province""",0,0.0
"""confidence""",0,0.0
"""frp""",0,0.0
"""bright_ti4""",0,0.0


## 3. Profil Statistik dengan DuckDB `SUMMARIZE`

DuckDB membaca Parquet langsung dari disk lalu menghitung min/max, jumlah nilai unik (perkiraan), rata-rata, standar deviasi, kuartil, dan persentase null untuk setiap kolom dalam satu perintah.

In [6]:
connection = duckdb.connect()
profil_duckdb = connection.execute(
    "SUMMARIZE SELECT * FROM read_parquet(?)", [str(RAW_PATH)]
).pl()
profil_duckdb

column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
str,str,str,str,i64,str,str,str,str,str,i64,"decimal[9,2]"
"""acq_date""","""TIMESTAMP_NS""","""2012-01-20 00:00:00""","""2026-10-27 00:00:00""",5255,"""2020-02-16 10:14:22.403661""",null,"""2015-09-22 01:55:34.664548""","""2019-09-15 09:03:18.654663""","""2024-09-25 20:03:35.452764""",6087130,0.00
"""acq_datetime""","""TIMESTAMP_NS""","""2012-01-20 03:57:00""","""2026-10-27 17:58:00""",147125,"""2020-02-16 20:17:35.484397""",null,"""2015-09-23 02:48:43.79169""","""2019-09-16 06:39:28.232002""","""2024-09-20 21:02:43.586626""",6087130,0.00
"""satellite""","""VARCHAR""","""NOAA-20""","""Suomi NPP""",2,null,null,null,null,null,6087130,0.00
"""latitude""","""DOUBLE""","""-10.97676""","""5.8782""",1269739,"""-3.0115979256431196""","""3.3751284721056076""","""-4.441222465930595""","""-2.642381117187363""","""-0.7084436952390681""",6087130,0.00
"""longitude""","""DOUBLE""","""95.01992""","""141.03944""",2579117,"""114.48423061182966""","""10.535289686092423""","""105.6909507970273""","""113.14514914742554""","""119.68812444004229""",6087130,0.00
"""province""","""VARCHAR""","""Aceh""","""West Sumatra""",30,null,null,null,null,null,6087130,0.00
"""confidence""","""VARCHAR""","""high""","""nominal""",3,null,null,null,null,null,6087130,0.00
"""frp""","""DOUBLE""","""0.0""","""1951.3""",16788,"""7.941055972190694""","""14.674035261411476""","""2.2757297995772445""","""4.490919958835068""","""8.314002356182167""",6087130,0.00
"""bright_ti4""","""DOUBLE""","""207.93""","""367.0""",6877,"""330.8085192660121""","""16.76002139267498""","""317.10211784510074""","""333.8006676350144""","""341.3581921493809""",6087130,0.00


## 4. Distribusi Kolom Kategorik

In [7]:
connection.execute(f"""
    CREATE OR REPLACE VIEW hotspot AS
    SELECT * FROM read_parquet('{RAW_PATH.as_posix()}')
""")

connection.execute("""
    SELECT satellite, processing,
           COUNT(*)                  AS jumlah,
           MIN(acq_date)::DATE       AS tanggal_awal,
           MAX(acq_date)::DATE       AS tanggal_akhir
    FROM hotspot
    GROUP BY ALL
    ORDER BY satellite, tanggal_awal
""").pl()

satellite,processing,jumlah,tanggal_awal,tanggal_akhir
str,str,i64,date,date
"""NOAA-20""","""SP""",1370633,2018-04-01,2026-07-27
"""NOAA-20""","""NRT""",575776,2026-07-28,2026-10-27
"""Suomi NPP""","""SP""",3574427,2012-01-20,2026-07-29
"""Suomi NPP""","""NRT""",566294,2026-07-30,2026-10-27


In [8]:
connection.execute("""
    SELECT confidence, daynight, COUNT(*) AS jumlah,
           ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS persen
    FROM hotspot
    GROUP BY ALL
    ORDER BY confidence, daynight
""").pl()

confidence,daynight,jumlah,persen
str,str,i64,f64
"""high""","""D""",170383,2.8
"""high""","""N""",24583,0.4
"""low""","""D""",369635,6.07
"""nominal""","""D""",3415173,56.1
"""nominal""","""N""",2107356,34.62


Distribusi per provinsi (34 provinsi). Hitungan hanya memakai **Suomi NPP**, karena dua satelit mengamati api yang sama dan tidak boleh dijumlahkan (lihat `data/README.md`).

In [9]:
connection.execute("""
    SELECT province,
           COUNT(*) AS deteksi_snpp,
           ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS persen,
           ROUND(MEDIAN(frp), 2) AS median_frp
    FROM hotspot
    WHERE satellite = 'Suomi NPP'
    GROUP BY province
    ORDER BY deteksi_snpp DESC
""").pl()

province,deteksi_snpp,persen,median_frp
str,i64,f64,f64
"""Central Kalimantan""",677588,16.36,4.17
"""West Kalimantan""",462542,11.17,5.97
"""South Sumatra""",441750,10.67,4.65
"""Riau""",397028,9.59,3.93
"""Papua""",318800,7.7,5.19
"""East Nusa Tenggara""",257099,6.21,4.6
"""East Kalimantan""",224464,5.42,4.07
"""South Kalimantan""",164923,3.98,4.5
"""Jambi""",141033,3.41,4.2


## 5. Cakupan Waktu

Jumlah deteksi per tahun per satelit dihitung dengan Polars lazy. Tahun 2026 baru sampai akhir Oktober.

In [10]:
per_tahun = (
    lazy_data
    .group_by(pl.col("acq_date").dt.year().alias("tahun"), "satellite")
    .agg(pl.len().alias("jumlah"))
    .collect()
    .pivot(on="satellite", index="tahun", values="jumlah")
    .sort("tahun")
)
per_tahun

tahun,NOAA-20,Suomi NPP
i32,u32,u32
2012,null,319942
2013,null,258195
2014,null,554115
2015,null,878049
2016,null,117425
2017,null,91267
2018,182372,192498
2019,438252,438278
2020,93944,92197


Hari tanpa satu pun deteksi pada rentang aktif tiap satelit. Ini bisa berarti memang tidak ada api, atau satelit sedang tidak beroperasi (*missing data* temporal). Akan diselidiki di Milestone 2.

In [11]:
connection.execute("""
    WITH hari AS (
        SELECT satellite, acq_date::DATE AS tanggal FROM hotspot GROUP BY ALL
    )
    SELECT satellite,
           MIN(tanggal) AS awal,
           MAX(tanggal) AS akhir,
           COUNT(*)     AS hari_ada_deteksi,
           date_diff('day', MIN(tanggal), MAX(tanggal)) + 1 - COUNT(*) AS hari_tanpa_deteksi
    FROM hari
    GROUP BY satellite
""").pl()

satellite,awal,akhir,hari_ada_deteksi,hari_tanpa_deteksi
str,date,date,i64,i64
"""Suomi NPP""",2012-01-20,2026-10-27,5340,55
"""NOAA-20""",2018-04-01,2026-10-27,3125,7


## 6. Pemeriksaan Awal Anomali

Daftar kandidat masalah kualitas yang akan ditangani pada Milestone 2.

In [12]:
anomali = connection.execute("""
    SELECT
        COUNT(*) AS total_baris,
        COUNT(*) - (SELECT COUNT(*) FROM (
            SELECT DISTINCT satellite, acq_datetime, latitude, longitude FROM hotspot
        ))                                                          AS duplikat_kunci,
        COUNT(*) FILTER (WHERE frp <= 0)                            AS frp_nol,
        COUNT(*) FILTER (WHERE bright_ti4 >= 367)                   AS ti4_saturasi,
        COUNT(*) FILTER (WHERE bright_ti5 <= 150)                   AS ti5_tidak_wajar,
        COUNT(*) FILTER (WHERE bright_ti4 < bright_ti5)             AS ti4_lebih_kecil_ti5,
        COUNT(*) FILTER (WHERE confidence = 'low')                  AS confidence_low,
        COUNT(*) FILTER (WHERE latitude NOT BETWEEN -11.5 AND 6.5
                            OR longitude NOT BETWEEN 94.5 AND 141.5) AS di_luar_indonesia
    FROM hotspot
""").pl()
anomali.transpose(include_header=True, header_name="pemeriksaan", column_names=["jumlah"])

pemeriksaan,jumlah
str,i64
"""total_baris""",6087130
"""duplikat_kunci""",0
"""frp_nol""",189
"""ti4_saturasi""",198472
"""ti5_tidak_wajar""",3
"""ti4_lebih_kecil_ti5""",6879
"""confidence_low""",369635
"""di_luar_indonesia""",0


In [13]:
connection.execute("""
    SELECT
        ROUND(MIN(frp), 2) AS min,
        ROUND(QUANTILE_CONT(frp, 0.25), 2) AS q1,
        ROUND(MEDIAN(frp), 2) AS median,
        ROUND(AVG(frp), 2) AS mean,
        ROUND(QUANTILE_CONT(frp, 0.75), 2) AS q3,
        ROUND(QUANTILE_CONT(frp, 0.99), 2) AS p99,
        ROUND(MAX(frp), 2) AS max,
        ROUND(SKEWNESS(frp), 2) AS skewness
    FROM hotspot
""").pl()

min,q1,median,mean,q3,p99,max,skewness
f64,f64,f64,f64,f64,f64,f64,f64
0.0,2.27,4.49,7.94,8.31,62.12,1951.3,14.16


## 7. Efisiensi: Eager vs Lazy vs DuckDB

Query yang sama — rata-rata FRP dan jumlah deteksi per tahun untuk Suomi NPP berkeyakinan `high` — dijalankan dengan tiga cara. Lazy dan DuckDB hanya membaca kolom yang diperlukan (*projection pushdown*) dan menyaring saat membaca (*predicate pushdown*).

In [14]:
def query_lazy():
    return (
        pl.scan_parquet(RAW_PATH)
        .filter((pl.col("satellite") == "Suomi NPP") & (pl.col("confidence") == "high"))
        .group_by(pl.col("acq_date").dt.year().alias("tahun"))
        .agg(pl.len().alias("jumlah"), pl.col("frp").mean().alias("rata_frp"))
        .sort("tahun")
    )

# Path absolut diganti "." agar output tidak bergantung lokasi folder
print(query_lazy().explain().replace(str(ROOT), "."))

SORT BY [col("tahun")]
  AGGREGATE[maintain_order: false]
    [len().alias("jumlah"), col("frp").mean().alias("rata_frp")] BY [col("acq_date").dt.year().alias("tahun")]
    FROM
    simple π 2/2 ["acq_date", "frp"]
      Parquet SCAN [./data/raw/hotspots.parquet]
      PROJECT 4/12 COLUMNS
      SELECTION: (col("confidence") == "high") & (col("satellite") == "Suomi NPP")
      ESTIMATED ROWS: 6087130


In [15]:
def ukur(fungsi, ulang=3):
    durasi = []
    for _ in range(ulang):
        mulai = time.perf_counter()
        hasil = fungsi()
        durasi.append(time.perf_counter() - mulai)
    return min(durasi), hasil


def eager():
    df = pl.read_parquet(RAW_PATH)  # seluruh kolom & baris masuk RAM
    hasil = (
        df.filter((pl.col("satellite") == "Suomi NPP") & (pl.col("confidence") == "high"))
        .group_by(pl.col("acq_date").dt.year().alias("tahun"))
        .agg(pl.len().alias("jumlah"), pl.col("frp").mean().alias("rata_frp"))
        .sort("tahun")
    )
    return hasil, df.estimated_size("mb")


def duck():
    return connection.execute("""
        SELECT year(acq_date) AS tahun, COUNT(*) AS jumlah, AVG(frp) AS rata_frp
        FROM hotspot
        WHERE satellite = 'Suomi NPP' AND confidence = 'high'
        GROUP BY tahun ORDER BY tahun
    """).pl()


t_eager, (hasil_eager, ram_eager) = ukur(eager)
t_lazy, hasil_lazy = ukur(lambda: query_lazy().collect())
t_duck, hasil_duck = ukur(duck)

# Ketiga metode harus memberi hasil yang sama (rata-rata dibandingkan dengan toleransi
# karena urutan penjumlahan floating point paralel bisa berbeda).
assert hasil_lazy.select("tahun", "jumlah").equals(hasil_eager.select("tahun", "jumlah"))
assert (hasil_lazy["rata_frp"] - hasil_eager["rata_frp"]).abs().max() < 1e-9
assert (hasil_lazy["rata_frp"] - hasil_duck["rata_frp"]).abs().max() < 1e-9
assert hasil_lazy.select("tahun", "jumlah").equals(
    hasil_duck.select(pl.col("tahun").cast(pl.Int32), pl.col("jumlah").cast(pl.UInt32))
)

benchmark = pl.DataFrame({
    "metode": ["Polars eager (read_parquet)", "Polars lazy (scan_parquet)", "DuckDB SQL"],
    "waktu_detik": [round(t_eager, 3), round(t_lazy, 3), round(t_duck, 3)],
})
print(f"DataFrame eager di RAM: ±{ram_eager:,.0f} MB")
benchmark

DataFrame eager di RAM: ±506 MB


metode,waktu_detik
str,f64
"""Polars eager (read_parquet)""",0.533
"""Polars lazy (scan_parquet)""",0.044
"""DuckDB SQL""",0.098


In [16]:
hasil_lazy

tahun,jumlah,rata_frp
i32,u32,f64
2012,10029,22.827833
2013,8755,23.884009
2014,20684,24.354823
2015,31723,26.090959
2016,2931,20.988492
2017,2139,17.215362
2018,6639,20.578196
2019,16676,23.946484
2020,2203,17.831325


## Catatan Milestone 1

**Ukuran dan struktur**
- 6.087.130 baris × 12 kolom, file Parquet 133 MB (memenuhi syarat > 1 juta baris). Jika dimuat penuh secara eager, DataFrame memakan ±506 MB RAM.
- Periode 20 Jan 2012 – 27 Okt 2026. Suomi NPP: 4,14 juta deteksi (sejak 2012). NOAA-20: 1,95 juta deteksi (sejak Apr 2018).
- Tipe data sudah sesuai (datetime, float, string), tetapi `acq_date` masih bertipe datetime padahal cukup `Date`. Kolom kategorik masih berupa `String`.

**Kelengkapan dan konsistensi**
- Tidak ada nilai null eksplisit di kolom mana pun, dan tidak ada duplikat pada kunci (satelit, waktu, lat, lon).
- Ada *missing* yang tersembunyi: 189 baris `frp = 0` (intensitas api tidak mungkin nol untuk deteksi api) dan 3 baris `bright_ti5 ≤ 150 K` (tidak realistis untuk permukaan bumi).
- Ada *missing* temporal: Suomi NPP tidak punya deteksi sama sekali pada 55 hari, NOAA-20 pada 7 hari. Perlu dicek apakah itu gangguan satelit atau memang tidak ada api.
- Data NRT (sementara) untuk ±3 bulan terakhir belum tumpang tindih dengan SP (arsip final). Meski begitu, pipeline tetap harus menjaga aturan "SP menggantikan NRT" karena dataset diperbarui harian.

**Validitas dan outlier**
- Semua koordinat berada di dalam kotak wilayah Indonesia dan semua baris punya provinsi (34 provinsi, nama bahasa Inggris).
- FRP sangat menceng ke kanan (median 4,5 MW, rata-rata 7,9 MW, maksimum 1.951 MW, skewness 14). Outlier perlu dideteksi pada skala log, dan nilai ekstrem kemungkinan kebakaran besar yang nyata, bukan error.
- 198.472 baris `bright_ti4 = 367 K` adalah batas saturasi sensor I4. Selain itu ada 6.879 baris dengan `bright_ti4 < bright_ti5` yang tidak lazim untuk piksel api. Keduanya perlu ditandai.
- 6% deteksi berkeyakinan `low`, dan semuanya terjadi pada siang hari. Ini kemungkinan pantulan matahari atau permukaan panas, sehingga perlu ditandai agar bisa disaring saat analisis.

**Gambaran awal**
- Lima provinsi teratas (Kalteng, Kalbar, Sumsel, Riau, Papua) menyumbang ±55% deteksi Suomi NPP.
- Tahun puncak: 2015 (878 ribu deteksi SNPP), 2019, dan **2026 (618 ribu deteksi, padahal baru sampai Oktober)**. Ini akan dianalisis di Milestone 3.

**Efisiensi:** untuk query agregasi yang sama, `scan_parquet` lazy (±0,04 dtk) dan DuckDB (±0,09 dtk) jauh lebih cepat daripada `read_parquet` eager (±0,55 dtk). Rencana query menunjukkan hanya 4 dari 12 kolom yang dibaca, dan filter diterapkan saat membaca file (*predicate pushdown*). Angka waktu bisa sedikit berbeda di tiap mesin.